# DATA 266 Lab 2 — Multimodal RAG (Kaggle)

**One-notebook run:** install → load competition data → auto-extract KB (tables + figure OCR) → write `submission.csv`.

### Before you run
1. **Add competition data** (PDFs + `Questions.csv` + `kb_manifest.json`) as the notebook input.
2. Turn on **GPU** (Accelerator) — speeds up EasyOCR.
3. Run **all cells** top to bottom.

No hand-coded answers: facts come from PDF table parsing + figure OCR.

**Output:** `/kaggle/working/submission.csv`

## 1) Install dependencies

Kaggle images usually already include `torch`. We install the packages needed for PDF parse + OCR + answering.

In [ ]:
# Minimal deps for the auto-extract submission path
!pip -q install pymupdf pandas rank-bm25 pillow easyocr opencv-python-headless

import sys, torch
print("Python:", sys.version.split()[0])
print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

## 2) Get the pipeline code

Clones the public Lab-2 repo into `/kaggle/working/repo` (or reuses it if already present).

If GitHub is blocked in your session, upload this same repo as a Kaggle Dataset and set `REPO_DIR` manually in the next cell.

In [ ]:
from pathlib import Path
import os, subprocess, sys

WORKING = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
REPO_DIR = WORKING / "repo"
REPO_URL = "https://github.com/Sarvesh0827/data266-lab2-multimodal-rag.git"

if (REPO_DIR / "rag").exists():
    print("Using existing repo at", REPO_DIR)
else:
    # Fresh clone of the pipeline
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)])
    print("Cloned", REPO_URL, "→", REPO_DIR)

sys.path.insert(0, str(REPO_DIR))
print("sys.path[0] =", sys.path[0])
assert (REPO_DIR / "rag" / "run_submission.py").exists(), "rag package missing — clone/upload failed"

## 3) Locate competition data

Finds `Questions.csv` under `/kaggle/input` (or local `data_export/` when testing offline).

In [ ]:
from pathlib import Path
import os

def find_data_dir() -> Path:
    # 1) Local / cloned data_export
    for cand in [REPO_DIR / "data_export", WORKING / "data_export", Path("data_export")]:
        if (cand / "Questions.csv").exists():
            return cand
    # 2) Kaggle competition / dataset inputs
    kaggle = Path("/kaggle/input")
    if kaggle.exists():
        hits = sorted(kaggle.rglob("Questions.csv"))
        if hits:
            return hits[0].parent
    raise FileNotFoundError(
        "Could not find Questions.csv. Add the competition data as a notebook input."
    )

DATA_DIR = find_data_dir()
os.environ["DATA_DIR"] = str(DATA_DIR)
# Keep artifacts + submission under /kaggle/working
os.environ["ARTIFACTS_DIR"] = str(WORKING / "artifacts")
os.environ["OUTPUT_DIR"] = str(WORKING)

print("DATA_DIR =", DATA_DIR)
print("PDFs:", sorted(p.name for p in DATA_DIR.glob("*.pdf")))
print("Questions:", (DATA_DIR / "Questions.csv").exists())
print("Manifest:", (DATA_DIR / "kb_manifest.json").exists())
assert (DATA_DIR / "Questions.csv").exists()
assert (DATA_DIR / "kb_manifest.json").exists()

## 4) Build KB + generate submission

This runs the full auto-extract pipeline:
- parse tables from PDFs
- OCR figures (A-4/A-5/A-6/A-7/B-4/B-5, …)
- answer all questions with statuses / citations / figure refs
- write `submission.csv`

In [ ]:
# Bind notebook env paths into rag.config, then run end-to-end
import importlib
import rag.config as cfg

cfg.refresh_paths()
print("config.DATA_DIR =", cfg.DATA_DIR)
print("config.ARTIFACTS =", cfg.ARTIFACTS)
print("config.OUTPUT_DIR =", cfg.OUTPUT_DIR)

# Ensure modules see refreshed paths (safe if re-running cells)
import rag.run_submission as rs
importlib.reload(rs)

out_path = WORKING / "submission.csv"
path = rs.run(
    questions_path=DATA_DIR / "Questions.csv",
    out_path=out_path,
    rebuild_index=False,
)
print("Wrote", path)
assert path.exists() and path.stat().st_size > 0

## 5) Validate + preview

In [ ]:
import pandas as pd
from pathlib import Path

sub = Path(path)
qs = DATA_DIR / "Questions.csv"

# Prefer competition validator if present
validator = DATA_DIR / "validate_submission.py"
if validator.exists():
    import subprocess, sys
    subprocess.check_call([sys.executable, str(validator), str(sub), str(qs)])
else:
    df = pd.read_csv(sub, dtype=str, keep_default_na=False)
    need = ["id", "conversation_id", "question_id", "answer", "figure_references", "citations", "status"]
    assert list(df.columns) == need or all(c in df.columns for c in need)
    assert len(df) == len(pd.read_csv(qs))
    print("basic validation ok:", len(df), "rows")

df = pd.read_csv(sub, dtype=str, keep_default_na=False)
print(df["status"].value_counts())
df.head(10)

## 6) Submit

In the Kaggle UI: **Submit to Competition** and choose `/kaggle/working/submission.csv`.

Optional: tag the notebook version (e.g. `p1-v1.0`) for the lab write-up.